# Simple keyframe OCR — one T4, Python 3.13
Attach your keyframe dataset, enable **Internet** and **GPU T4 x2**, edit only `INPUT_ROOT` below, then **Run All**. Only physical GPU 0 is used. No CPU fallback. Installation and model downloads require Internet; the first run can take several minutes.
Input examples: `/kaggle/input/my-keyframes/L04` (contains `L04_V001/001.jpg`), or `/kaggle/input/my-keyframes/L04/L04_V001`. Only the selected subtree is read. Numeric filenames mean **keyframe_n**, not video frame indices; exported `frame_id` is null. No caption mapping is read or inferred.
The wrapper adds 32 white pixels on every side, without resizing or EXIF rotation. Paddle's detector and recognizer still perform their own internal resizing/cropping. Latin recognition includes Vietnamese. Technical **simple-kaggle-ocr-v1** checkpoints remain private; downloads contain only the seven-field public projection, not ocr-sidecar-v1.
## 1 · Input (the only setting to edit)

In [ ]:
INPUT_ROOT = '/kaggle/input/your-keyframes/L04'
RUN_OK = False  # Always invalidate download links when starting again.
EXPORT_OK = False


## 2 · Isolated dependencies and readable GPU worker
No packages are installed into the notebook kernel. A pip-less venv uses the host's pip `--python` option (no ensurepip). Exact core pins, binary wheels only; dependency failures stop, without unpinning or compiling. The environment is reused, never deleted.
Evidence: [official CUDA 12.6 cp313 wheel index](https://www.paddlepaddle.org.cn/packages/stable/cu126/paddlepaddle-gpu/), [PaddleOCR 3.2.0 metadata](https://pypi.org/pypi/paddleocr/3.2.0/json), [PaddleX 3.2.0 metadata](https://pypi.org/pypi/paddlex/3.2.0/json), [NumPy 2.2.6 cp313 wheels](https://pypi.org/pypi/numpy/2.2.6/json). The [versioned OCR API](https://github.com/PaddlePaddle/PaddleOCR/blob/v3.2.0/paddleocr/_pipelines/ocr.py) maps Vietnamese to `latin_PP-OCRv5_mobile_rec`; [common GPU flags](https://github.com/PaddlePaddle/PaddleOCR/blob/v3.2.0/paddleocr/_common_args.py) support `device`, `enable_hpi`, and `use_tensorrt`. This combination is metadata-checked, **not yet validated on Kaggle**.
The source below is ordinary Python, written to a transient worker file so OCR runs in the isolated interpreter with live stdout/stderr. It has no repository imports. Notebook-managed runtime, known library cache/config paths, checkpoints and loose exports use an owned `/kaggle/temp` directory (guarded writable `/tmp` fallback), not `/kaggle/working`. Children inherit only explicitly allowed system/driver/network settings; TLS key logging and cuDNN file-log overrides are excluded, glog uses stderr, and Matplotlib config uses scratch. This is filesystem hygiene for the pinned stack, not a sandbox for arbitrary external code. No existing environment or unrelated output is deleted.

In [ ]:
import hashlib
import json
import os
from pathlib import Path
import subprocess
import sys
import uuid

RUN_OK = False
EXPORT_OK = False
WORKER_SOURCE = r'''
import argparse
import hashlib
import importlib.metadata
import io
import json
import math
import os
from pathlib import Path
import re
import shutil
import stat
import sys
import traceback
import uuid
import zipfile

SCHEMA = 'simple-kaggle-ocr-v1'
PINS = {'paddlepaddle-gpu': '3.2.2', 'paddleocr': '3.2.0',
        'paddlex': '3.2.0', 'numpy': '2.2.6', 'Pillow': '11.3.0',
        'tqdm': '4.67.1', 'opencv-contrib-python': '4.10.0.84'}
MODELS = {'detection': 'PP-OCRv5_server_det',
          'recognition': 'latin_PP-OCRv5_mobile_rec'}
OPTIONS = dict(device='gpu:0', enable_hpi=False, use_tensorrt=False,
               text_detection_model_name=MODELS['detection'],
               text_recognition_model_name=MODELS['recognition'],
               use_doc_orientation_classify=False, use_doc_unwarping=False,
               use_textline_orientation=False, text_det_limit_side_len=960,
               text_det_limit_type='max', text_det_thresh=0.3,
               text_det_box_thresh=0.6, text_det_unclip_ratio=1.5,
               text_rec_score_thresh=0.0, text_recognition_batch_size=1)


def digest(raw):
    return hashlib.sha256(raw).hexdigest()


def encoded(value):
    return json.dumps(value, ensure_ascii=False, sort_keys=True,
                      separators=(',', ':'), allow_nan=False).encode('utf-8')


def safe_path(value):
    path = Path(value)
    if not path.is_absolute() or '..' in path.parts:
        raise ValueError('Use absolute paths without traversal')
    for part in [path, *path.parents]:
        try:
            info = part.lstat()
        except FileNotFoundError:
            continue
        if stat.S_ISLNK(info.st_mode) or getattr(info, 'st_file_attributes', 0) & 0x400:
            raise ValueError(f'Symlink/reparse path rejected: {part}')
    return path.resolve()


def safe_tree(root):
    root = safe_path(root)
    if root.exists():
        for folder, dirs, files in os.walk(root, followlinks=False):
            if len(Path(folder).relative_to(root).parts) > 8:
                raise ValueError('Directory nesting exceeds 8 levels')
            for name in dirs + files:
                safe_path(Path(folder) / name)
    return root


def safe_runtime_tree(root):
    # venv lib64 and downloaded model caches may have generated INTERNAL links.
    # Never allow an existing runtime link to redirect pip/cache writes outside it.
    root = safe_path(root)
    for folder, dirs, files in os.walk(root, followlinks=False):
        for name in dirs + files:
            path = Path(folder) / name
            info = path.lstat()
            if stat.S_ISLNK(info.st_mode) or getattr(info, 'st_file_attributes', 0) & 0x400:
                target = path.resolve()
                if target == root or root not in target.parents:
                    raise ValueError(f'External runtime symlink/reparse path rejected: {path}')


def validate_paths(input_root, output, env_root):
    paths = [safe_path(p) for p in (input_root, output, env_root)]
    for i, first in enumerate(paths):
        for second in paths[i + 1:]:
            if first == second or first in second.parents or second in first.parents:
                raise ValueError('INPUT_ROOT, output and venv must be disjoint in BOTH directions')
    safe_tree(paths[0])
    safe_tree(paths[1])
    safe_runtime_tree(paths[2])
    if not paths[0].is_dir():
        raise ValueError('INPUT_ROOT must be an existing directory')
    return paths


COPY_SUFFIX = re.compile(r'\s*\(\d+\)\s*$')


def inventory(root):
    root = safe_tree(root)
    items, seen, export_paths = [], set(), set()
    for path in sorted(root.rglob('*')):
        if not path.is_file() or path.suffix.lower() not in {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tif', '.tiff'}:
            continue
        # Skip copy-style names such as '001 (1).jpg' (also inside folders like 'L04_V001 (1)').
        if COPY_SUFFIX.search(path.stem) or any(COPY_SUFFIX.search(p) for p in path.relative_to(root).parts[:-1]):
            continue
        match = re.fullmatch(r'L([0-9]{1,4})_V([0-9]{1,6})', path.parent.name, re.I)
        if not match or not re.fullmatch(r'[0-9]{1,12}', path.stem):
            raise ValueError(f'Expected video folder / numeric keyframe filename: {path}')
        video = f'L{int(match[1]):02d}_V{int(match[2]):03d}'
        if path.parent.name != video:
            raise ValueError(f'Expected canonical video folder {video}: {path}')
        number = int(path.stem)
        identity = f'{video}:{number}'
        group = video.split('_V')[0]
        export_path = f'{group}/{video}/{path.stem}.json'
        if identity in seen:
            raise ValueError(f'Duplicate canonical identity: {identity}')
        if export_path in export_paths:
            raise ValueError(f'Duplicate export path: {export_path}')
        seen.add(identity)
        export_paths.add(export_path)
        items.append(dict(id=identity, video_id=video, keyframe_n=number,
                          relative_path=path.relative_to(root).as_posix()))
    if not items:
        raise ValueError('No keyframe images inside INPUT_ROOT')
    return sorted(items, key=lambda x: (x['video_id'], x['keyframe_n']))


def atomic_bytes(path, raw):
    path = safe_path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + '.' + uuid.uuid4().hex + '.tmp')
    created = False
    try:
        with temporary.open('xb') as stream:
            created = True
            stream.write(raw)
            stream.flush()
            os.fsync(stream.fileno())
        os.replace(temporary, path)
    finally:
        if created:
            temporary.unlink(missing_ok=True)


def atomic_json(path, value):
    atomic_bytes(path, encoded(value) + b'\n')


def read_json(path):
    return json.loads(safe_path(path).read_text(encoding='utf-8'))


def claim_output(root, output, create=True):
    owner = {'schema': SCHEMA, 'input_root': str(root)}
    marker = output / 'owner.json'
    if output.exists() and any(output.iterdir()):
        if not marker.is_file() or read_json(marker) != owner:
            raise ValueError('Refusing output owned by another dataset or unrelated files')
    elif create:
        atomic_json(marker, owner)


def runtime_identity(env_root):
    marker = env_root / 'simple-runtime.json'
    expected = {'schema': 'simple-kaggle-runtime-v1', 'pins': PINS}
    if env_root.exists() and any(env_root.iterdir()):
        if not marker.is_file() or read_json(marker) != expected:
            raise RuntimeError('Unowned/incompatible venv; refusing to modify or delete it')
    return marker, expected


class PublicationCollision(ValueError):
    pass


def validate_publish(root, output, env_root, archive):
    root, output, env_root = validate_paths(root, output, env_root)
    archive = safe_path(archive)
    for path in (root, output, env_root):
        if (path == archive or path in archive.parents or archive in path.parents
                or archive.parent == path or archive.parent in path.parents):
            raise ValueError('Publish directory and input/scratch must be disjoint in BOTH directions')
    if archive.exists():
        receipt = output / 'publication.json'
        if not receipt.is_file():
            raise PublicationCollision('Unrelated preexisting ocr.zip; refusing overwrite')
        owned = read_json(receipt)
        if (owned.get('archive') != str(archive) or owned.get('input_root') != str(root)
                or owned.get('archive_sha256') != file_digest(archive)):
            raise PublicationCollision('Unrelated or changed preexisting ocr.zip; refusing overwrite')
    return archive


def file_digest(path):
    value = hashlib.sha256()
    with safe_path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            value.update(block)
    return value.hexdigest()


def writable_probe(folder):
    folder = safe_path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    probe = folder / ('.write-probe-' + uuid.uuid4().hex)
    created = False
    try:
        with probe.open('xb') as stream:
            created = True
            stream.write(b'probe')
            stream.flush()
            os.fsync(stream.fileno())
    finally:
        if created:
            probe.unlink(missing_ok=True)


def select_scratch(root, archive, candidates=('/kaggle/temp', '/tmp')):
    root = safe_tree(root)
    key = digest(str(root).encode())
    owner = dict(schema='simple-kaggle-scratch-v2', input_root=str(root))
    failures, collisions = [], []
    for candidate in candidates:
        scratch = safe_path(Path(candidate) / ('ocr-simple-gpu-' + key))
        if root == scratch or root in scratch.parents or scratch in root.parents:
            raise ValueError('Scratch and input must be disjoint in BOTH directions')
        output, runtime = scratch / 'state', scratch / 'runtime'
        # Validate every destination and ownership BEFORE even a permission probe.
        try:
            validate_publish(root, output, runtime, archive)
        except PublicationCollision as exc:
            # A previous fallback profile may own the ZIP; never write here.
            collisions.append(str(exc))
            continue
        safe_runtime_tree(scratch)
        marker = scratch / 'scratch-owner.json'
        if scratch.exists() and any(scratch.iterdir()):
            if not marker.is_file() or read_json(marker) != owner:
                raise ValueError('Unowned scratch; refusing to modify or delete it')
        claim_output(root, output, create=False)
        runtime_identity(runtime)
        try:
            writable_probe(scratch)
        except OSError as exc:
            failures.append(f'{scratch}: {exc}')
            continue
        if not marker.exists():
            atomic_json(marker, owner)
        claim_output(root, output)
        return scratch, output, runtime
    if collisions:
        raise PublicationCollision('Unrelated preexisting ocr.zip or unavailable owning scratch; refusing overwrite')
    raise RuntimeError('No writable transient scratch (never using working): ' + '; '.join(failures))


def isolated_environment(scratch, inherited):
    scratch = safe_path(scratch)
    # Inherit capabilities, not arbitrary cache/log/profiling write destinations.
    allowed = {'PATH', 'SYSTEMROOT', 'SystemRoot', 'WINDIR', 'COMSPEC', 'PATHEXT',
               'LANG', 'LC_ALL', 'LC_CTYPE', 'TZ', 'TERM', 'NO_COLOR',
               'LD_LIBRARY_PATH', 'CUDA_HOME', 'CUDA_PATH', 'CUDA_MODULE_LOADING',
               'CUDA_DEVICE_MAX_CONNECTIONS', 'CUDA_LAUNCH_BLOCKING',
               'NVIDIA_VISIBLE_DEVICES', 'NVIDIA_DRIVER_CAPABILITIES',
               'OMP_NUM_THREADS', 'MKL_NUM_THREADS', 'OPENBLAS_NUM_THREADS',
               'HTTP_PROXY', 'HTTPS_PROXY', 'ALL_PROXY', 'NO_PROXY',
               'http_proxy', 'https_proxy', 'all_proxy', 'no_proxy',
               'SSL_CERT_FILE', 'SSL_CERT_DIR', 'REQUESTS_CA_BUNDLE', 'CURL_CA_BUNDLE',
               'HF_TOKEN', 'HUGGING_FACE_HUB_TOKEN', 'MODELSCOPE_API_TOKEN'}
    environment = {k: v for k, v in inherited.items() if k in allowed}
    routes = {'HOME': 'home', 'USERPROFILE': 'home', 'XDG_CACHE_HOME': 'cache',
              'XDG_CONFIG_HOME': 'config', 'XDG_DATA_HOME': 'data', 'XDG_STATE_HOME': 'local-state',
              'PIP_CACHE_DIR': 'cache/pip', 'HF_HOME': 'cache/huggingface',
              'HF_HUB_CACHE': 'cache/huggingface/hub',
              'HUGGINGFACE_HUB_CACHE': 'cache/huggingface/hub',
              'HF_ASSETS_CACHE': 'cache/huggingface/assets',
              'HUGGINGFACE_ASSETS_CACHE': 'cache/huggingface/assets',
              'TRANSFORMERS_CACHE': 'cache/huggingface/hub',
              'PYTORCH_TRANSFORMERS_CACHE': 'cache/huggingface/hub',
              'PYTORCH_PRETRAINED_BERT_CACHE': 'cache/huggingface/hub',
              'PADDLE_HOME': 'cache/paddle', 'PADDLE_OCR_BASE_DIR': 'cache/paddleocr',
              'PADDLE_PDX_CACHE_HOME': 'cache/paddlex', 'PADDLEX_HOME': 'cache/paddlex',
              'MODELSCOPE_CACHE': 'cache/modelscope', 'TORCH_HOME': 'cache/torch',
              'CUDA_CACHE_PATH': 'cache/cuda', 'MPLCONFIGDIR': 'config/matplotlib',
              'TMPDIR': 'tmp', 'TEMP': 'tmp', 'TMP': 'tmp'}
    for key, relative in routes.items():
        path = safe_path(scratch / relative)
        path.mkdir(parents=True, exist_ok=True)
        environment[key] = str(path)
    environment.update(CUDA_VISIBLE_DEVICES='0', CUDA_DEVICE_ORDER='PCI_BUS_ID',
                       PIP_CONFIG_FILE=os.devnull, PYTHONNOUSERSITE='1', PYTHONUNBUFFERED='1',
                       GLOG_logtostderr='1')
    return environment


def invalidate_ready(output):
    safe_path(output / 'ready.json').unlink(missing_ok=True)


def invalidate_current_ready(root, output, env_root, run_id):
    root, output, env_root = validate_paths(root, output, env_root)
    claim_output(root, output, create=False)
    path = safe_path(output / 'ready.json')
    try:
        ready = read_json(path)
    except FileNotFoundError:
        return
    if (isinstance(ready, dict) and ready.get('schema') == SCHEMA
            and ready.get('input_root') == str(root) and ready.get('run_id') == run_id):
        path.unlink()  # Never remove publication ownership or another invocation's receipt.


def checked_ready(root, output, env_root, run_id, archive):
    archive = validate_publish(root, output, env_root, archive)
    ready = read_json(output / 'ready.json')
    state = read_json(output / 'run.json')
    if (ready.get('run_id') != run_id or state.get('run_id') != run_id
            or state.get('phase') != 'finished'):
        raise RuntimeError('Export invocation mismatch')
    if (ready.get('profile_sha256') != state.get('profile_sha256')
            or digest(encoded(read_json(output / 'profile.json'))) != ready.get('profile_sha256')
            or ready != read_json(output / 'publication.json')
            or ready.get('archive_sha256') != file_digest(archive)):
        raise RuntimeError('Export receipt/digest mismatch')
    return archive


def publish_zip(root, output, env_root, run_id, source, archive, summary):
    archive = validate_publish(root, output, env_root, archive)
    archive.parent.mkdir(parents=True, exist_ok=True)
    temporary = archive.parent / ('.ocr-publish-' + uuid.uuid4().hex + '.tmp')
    created = False
    try:
        # Copy, not rename: scratch and working may be on different filesystems.
        with temporary.open('xb') as target:
            created = True
            with source.open('rb') as stream:
                shutil.copyfileobj(stream, target)
            target.flush()
            os.fsync(target.fileno())
        if file_digest(temporary) != file_digest(source):
            raise RuntimeError('Publication copy integrity failure')
        with zipfile.ZipFile(temporary) as check:
            if check.testzip() is not None:
                raise RuntimeError('Invalid publication ZIP')
        receipt = dict(summary, archive=str(archive), archive_sha256=file_digest(temporary))
        os.replace(temporary, archive)
        atomic_json(output / 'publication.json', receipt)
        atomic_json(output / 'ready.json', receipt)  # Scratch only; written last.
    finally:
        if created:
            temporary.unlink(missing_ok=True)  # Only this invocation's own file.
    return archive


def decode(raw):
    from PIL import Image, ImageOps
    # Deliberately no exif_transpose: coordinates refer to the stored raster.
    with Image.open(io.BytesIO(raw)) as source:
        source.load()  # Force lazy decoder exceptions into the decode-only handler.
        image = source.convert('RGB')
    return ImageOps.expand(image, border=32, fill='white'), image.size


def decode_or_error(raw):
    from PIL import Image
    try:
        return decode(raw), None
    except (OSError, SyntaxError, ValueError, Image.DecompressionBombError) as exc:
        return None, {'type': type(exc).__name__, 'message': str(exc)}


def require_gpu(paddle):
    if os.environ.get('CUDA_VISIBLE_DEVICES') != '0':
        raise RuntimeError('CUDA_VISIBLE_DEVICES must be exactly 0 before importing paddle')
    if not paddle.is_compiled_with_cuda() or paddle.device.cuda.device_count() != 1:
        raise RuntimeError('A CUDA build and exactly one visible GPU are required; no CPU fallback')
    name = paddle.device.cuda.get_device_name(0)
    if 'T4' not in name:
        raise RuntimeError(f'Expected selected NVIDIA T4, got {name}')
    paddle.set_device('gpu:0')
    value = paddle.to_tensor([[2.0]], place=paddle.CUDAPlace(0))
    result = paddle.matmul(value, value)
    paddle.device.cuda.synchronize()
    if not result.place.is_gpu_place() or float(result.numpy()[0][0]) != 4.0:
        raise RuntimeError('Actual CUDA tensor operation failed')
    print(f'CUDA_VISIBLE_DEVICES=0; physical GPU 0 -> gpu:0: {name}; CUDA tensor smoke OK', flush=True)
    return {'device': 'gpu:0', 'physical_gpu': 0, 'gpu_name': name,
            'cuda': paddle.version.cuda(), 'cudnn': paddle.version.cudnn()}


class Adapter:
    def __init__(self):
        os.environ['CUDA_VISIBLE_DEVICES'] = '0'
        import paddle
        gpu = require_gpu(paddle)
        versions = {name: importlib.metadata.version(name) for name in PINS}
        if versions != PINS:
            raise RuntimeError(f'Core version mismatch: {versions}; expected {PINS}')
        from paddleocr import PaddleOCR
        self.ocr = PaddleOCR(**OPTIONS)
        pipe = self.ocr.paddlex_pipeline
        models = {'detection': pipe.text_det_model.model_name,
                  'recognition': pipe.text_rec_model.model_name}
        if models != MODELS:
            raise RuntimeError(f'Unexpected actual models: {models}')
        for model in (pipe.text_det_model, pipe.text_rec_model):
            if model.pp_option.device_type != 'gpu' or model.pp_option.device_id != 0:
                raise RuntimeError('OCR predictor is not on gpu:0')
        self.engine = dict(gpu, models=models, core_versions=versions,
                           packages=dict(sorted((d.metadata['Name'], d.version)
                                                for d in importlib.metadata.distributions())))
        print(json.dumps(self.engine, ensure_ascii=False, indent=2), flush=True)

    def recognize(self, padded):
        import numpy as np
        # PaddleX's ndarray reader expects BGR, not Pillow RGB.
        results = self.ocr.predict(input=np.asarray(padded)[:, :, ::-1].copy())
        if len(results) != 1 or 'error' in results[0]:
            raise RuntimeError(f'Unexpected OCR result: {results}')
        result = results[0]
        texts, scores, polygons = (result[k] for k in ('rec_texts', 'rec_scores', 'rec_polys'))
        if not len(texts) == len(scores) == len(polygons):
            raise RuntimeError('OCR API returned inconsistent line arrays')
        return [dict(text=text, confidence=float(score), polygon_padded=poly.tolist())
                for text, score, poly in zip(texts, scores, polygons)]


def make_profile(adapter):
    profile = dict(schema=SCHEMA, engine=adapter.engine, options=OPTIONS,
                   padding_px=32, wrapper_resize=False, exif_rotation=False,
                   worker_sha256=digest(Path(__file__).read_bytes()))
    return profile, digest(encoded(profile))


def record_for(item, sha, profile_hash, decoded, error, adapter):
    record = dict(item, schema=SCHEMA, source_sha256=sha, profile_sha256=profile_hash,
                  mapping=None, status='error' if error else 'empty', error=error, lines=[],
                  geometry=None)
    if error:
        return record
    padded, (width, height) = decoded
    record['geometry'] = dict(original_size=[width, height], padded_size=list(padded.size),
                              padding_ltrb=[32, 32, 32, 32], wrapper_resize=False,
                              original_to_padded=[[1, 0, 32], [0, 1, 32], [0, 0, 1]],
                              padded_to_original=[[1, 0, -32], [0, 1, -32], [0, 0, 1]])
    for line in adapter.recognize(padded):
        polygon = [[float(x), float(y)] for x, y in line['polygon_padded']]
        score = float(line['confidence'])
        if len(polygon) != 4 or not isinstance(line['text'], str) or not 0 <= score <= 1:
            raise RuntimeError('Invalid OCR line API values')
        if not all(math.isfinite(v) for point in polygon for v in point):
            raise RuntimeError('Non-finite polygon')
        original = [[x - 32, y - 32] for x, y in polygon]
        clipped = [[max(0, min(width, x)), max(0, min(height, y))] for x, y in original]
        record['lines'].append(dict(text=line['text'], confidence=score,
                                    polygon_padded=polygon, polygon_original_unclipped=original,
                                    polygon_original_clipped=clipped))
    record['status'] = 'ok' if record['lines'] else 'empty'
    return record


def checkpoint_path(output, item):
    return output / 'checkpoints' / (digest(item['id'].encode()) + '.json')


def save_checkpoint(output, record):
    atomic_json(checkpoint_path(output, record),
                {'record': record, 'record_sha256': digest(encoded(record))})


def cached(output, item, sha, profile_hash):
    path = checkpoint_path(output, item)
    if not path.exists():
        return None
    try:
        envelope = read_json(path)
        record = envelope['record']
        if envelope['record_sha256'] != digest(encoded(record)):
            return None
        expected = dict(item, schema=SCHEMA, source_sha256=sha, profile_sha256=profile_hash, mapping=None)
        if any(record[k] != value for k, value in expected.items()):
            return None
        if record['status'] not in {'ok', 'empty', 'error'} or not isinstance(record['lines'], list):
            return None
        if record['status'] in {'ok', 'empty'}:
            if record['error'] is not None or not record['geometry']:
                return None
            if (record['status'] == 'ok') != bool(record['lines']):
                return None
        if record['status'] == 'error' and (not record['error'] or record['lines']):
            return None
        return record
    except (ValueError, KeyError, TypeError):
        print(f'Invalid checkpoint, recompute: {path.name}', flush=True)
        return None


def run(root, output, env_root, run_id, adapter=None, progress=None):
    root, output, env_root = validate_paths(root, output, env_root)
    claim_output(root, output)
    state = {'run_id': run_id, 'phase': 'running', 'complete': False}
    invalidate_ready(output)
    atomic_json(output / 'run.json', state)
    try:
        items = inventory(root)  # Reject duplicates before model load or bulk work.
        adapter = adapter if adapter is not None else Adapter()
        profile, profile_hash = make_profile(adapter)
        # A real decodable input, the SAME model/adapter/padding path as bulk.
        smoke = None
        for item in items:
            raw = safe_path(root / item['relative_path']).read_bytes()
            decoded, error = decode_or_error(raw)
            if not error:
                smoke = record_for(item, digest(raw), profile_hash, decoded, None, adapter)
                break
        if smoke is None:
            raise RuntimeError('No decodable real image for mandatory GPU OCR smoke')
        print(f'Real image OCR smoke OK: {smoke["relative_path"]}; profile={profile_hash}', flush=True)
        atomic_json(output / 'profile.json', profile)
        if progress is None:
            from tqdm import tqdm
            progress = lambda rows: tqdm(rows, total=len(rows), unit='image',
                                         desc='OCR (including resume checks)', file=sys.stdout,
                                         mininterval=1, smoothing=0.1)
        errors, skipped = 0, 0
        for item in progress(items):
            raw = safe_path(root / item['relative_path']).read_bytes()
            sha = digest(raw)
            old = cached(output, item, sha, profile_hash)
            if old and old['status'] in {'ok', 'empty'}:
                skipped += 1
                continue
            if item['id'] == smoke['id'] and sha == smoke['source_sha256']:
                record = smoke
            else:
                decoded, error = decode_or_error(raw)
                # Backend exceptions are intentionally OUTSIDE the decode handler.
                record = record_for(item, sha, profile_hash, decoded, error, adapter)
            save_checkpoint(output, record)  # fsync + atomic replace after every image.
            if record['status'] == 'error':
                errors += 1
                print(f'Image decode error: {item["relative_path"]}: {record["error"]}', flush=True)
        state.update(phase='finished', complete=errors == 0, errors=errors,
                     skipped=skipped, total=len(items), profile_sha256=profile_hash, inventory=items)
        atomic_json(output / 'run.json', state)
        print(f'Finished: {len(items)} images; resumed {skipped}; decode errors {errors}', flush=True)
    except BaseException as exc:
        state.update(phase='fatal', complete=False, error=f'{type(exc).__name__}: {exc}')
        atomic_json(output / 'run.json', state)
        traceback.print_exc()
        raise


def public_record(record):
    # Project only integrity-verified checkpoints; preserve model text and order.
    text = '' if record['status'] == 'error' else '\n'.join(
        line['text'] for line in record['lines'] if line['text'].strip())
    video = record['video_id']
    group = video.split('_V')[0]
    return dict(video_id=video, frame_id=None, keyframe_n=record['keyframe_n'],
                image_path=f'data/{group}/{video}/{Path(record["relative_path"]).name}',
                ocr_text=text, retrieval_text=text,
                status='error' if record['status'] == 'error' else ('ok' if text else 'empty'))


def export_current(root, output, env_root, run_id, archive='/kaggle/working/ocr.zip'):
    root, output, env_root = validate_paths(root, output, env_root)
    archive = validate_publish(root, output, env_root, archive)
    claim_output(root, output)
    invalidate_ready(output)
    state = read_json(output / 'run.json')
    if state['run_id'] != run_id or state['phase'] != 'finished':
        raise RuntimeError('No successful current invocation; refusing stale downloads')
    profile = read_json(output / 'profile.json')
    if digest(encoded(profile)) != state['profile_sha256']:
        raise RuntimeError('Profile integrity failure')
    items = inventory(root)
    if items != state['inventory']:
        raise RuntimeError('Input inventory changed since run; run cell 3 again')
    records, missing = [], []
    for item in items:
        sha = digest(safe_path(root / item['relative_path']).read_bytes())
        record = cached(output, item, sha, state['profile_sha256'])
        if record is None:
            missing.append(item['id'])
        else:
            records.append(record)
    if len({r['id'] for r in records}) != len(records):
        raise RuntimeError('Duplicate final IDs')
    public_records = [public_record(record) for record in records]
    errors = sum(r['status'] == 'error' for r in public_records)
    summary = dict(schema=SCHEMA, run_id=run_id, input_root=str(root),
                   total=len(items), exported=len(records), errors=errors, missing=missing,
                   ok=sum(r['status'] == 'ok' for r in public_records),
                   empty=sum(r['status'] == 'empty' for r in public_records),
                   complete=not errors and not missing, profile_sha256=state['profile_sha256'])
    # Fresh invocation directory: never serve old per-image JSON/ZIP after a failure.
    destination = output / 'downloads' / run_id
    if destination.exists():
        raise RuntimeError('Export already exists; rerun cell 3 for a fresh export')
    names = []
    for record, public in zip(records, public_records):
        # Inventory validated canonical folders, numeric stems and collisions before writes.
        video = record['video_id']
        group = video.split('_V')[0]  # Derive from identity, never input ancestors.
        name = f'{group}/{video}/{Path(record["relative_path"]).stem}.json'
        atomic_bytes(destination / name, (json.dumps([public], ensure_ascii=False, indent=2,
                                                    allow_nan=False) + '\n').encode('utf-8'))
        names.append(name)
    temporary = destination / 'ocr.zip.tmp'
    with zipfile.ZipFile(temporary, 'x', compression=zipfile.ZIP_DEFLATED) as packed:
        # Explicit allowlist: ONLY per-image OCR records, nothing else.
        for name in names:
            packed.write(destination / name, name)
    with zipfile.ZipFile(temporary) as check:
        if check.namelist() != names or check.testzip() is not None:
            raise RuntimeError('Export ZIP integrity failure')
    os.replace(temporary, destination / 'ocr.zip')
    try:
        publish_zip(root, output, env_root, run_id, destination / 'ocr.zip', archive, summary)
        print(json.dumps(summary, indent=2), flush=True)  # Log only; not in the ZIP.
        print(f'Download: {archive}', flush=True)
        return destination
    except BaseException:
        invalidate_current_ready(root, output, env_root, run_id)
        raise


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('action', choices=['run', 'export'])
    parser.add_argument('input_root')
    parser.add_argument('output')
    parser.add_argument('env_root')
    parser.add_argument('run_id')
    parser.add_argument('--archive', default='/kaggle/working/ocr.zip')
    args = parser.parse_args()
    if not re.fullmatch(r'[a-f0-9]{32}', args.run_id):
        raise ValueError('Invalid invocation ID')
    validate_publish(args.input_root, args.output, args.env_root, args.archive)
    if args.action == 'run':
        run(args.input_root, args.output, args.env_root, args.run_id)
    else:
        export_current(args.input_root, args.output, args.env_root, args.run_id, args.archive)


if __name__ == '__main__':
    main()
'''

# Load just stdlib path/inventory helpers, never Paddle in the kernel.
helpers = {'__name__': 'notebook_helpers'}
exec(compile(WORKER_SOURCE, '<visible-worker>', 'exec'), helpers)
root = helpers['safe_path'](INPUT_ROOT)
ARCHIVE = Path('/kaggle/working/ocr.zip')
helpers['inventory'](root)
if sys.version_info[:2] != (3, 13) or sys.platform != 'linux':
    raise RuntimeError('This notebook requires Kaggle Linux Python 3.13')
SCRATCH, OUTPUT, ENV_ROOT = helpers['select_scratch'](root, ARCHIVE)
helpers['invalidate_ready'](OUTPUT)
runtime_owner, expected_owner = helpers['runtime_identity'](ENV_ROOT)
if not runtime_owner.exists():
    helpers['atomic_json'](runtime_owner, expected_owner)
# Check runtime write permission before invoking either installer.
helpers['writable_probe'](ENV_ROOT)
child_env = helpers['isolated_environment'](SCRATCH, os.environ)
PYTHON = ENV_ROOT / 'bin/python'
if not PYTHON.exists():
    subprocess.run([sys.executable, '-I', '-m', 'venv', '--without-pip', '--copies', str(ENV_ROOT)],
                   env=child_env, cwd=SCRATCH, check=True)
gpu_wheel = ('https://paddle-whl.cdn.bcebos.com/stable/cu126/paddlepaddle-gpu/'
             'paddlepaddle_gpu-3.2.2-cp313-cp313-linux_x86_64.whl')
requirements = [f'{name}=={version}' for name, version in helpers['PINS'].items()
                if name != 'paddlepaddle-gpu']
pip_command = [sys.executable, '-I', '-m', 'pip', '--python', str(PYTHON), '--isolated',
               'install', '--only-binary=:all:', '--no-cache-dir',
               '--index-url', 'https://pypi.org/simple',
               gpu_wheel, *requirements]
print('Installing pinned wheels into', ENV_ROOT, '(kernel packages unchanged)', flush=True)
subprocess.run(pip_command, env=child_env, cwd=SCRATCH, check=True)
# Notebook-managed files and known library cache defaults use transient scratch.
WORKER = OUTPUT / 'worker.py'
helpers['atomic_bytes'](WORKER, WORKER_SOURCE.encode('utf-8'))
print('Output:', OUTPUT, '\nSelected GPU: physical 0 only; runtime verification follows in cell 3.', flush=True)


## 3 · GPU verification, real-image smoke, OCR and resume
A CUDA build, one visible T4, a CUDA matrix operation, GPU predictor settings, and OCR on a real padded input must all succeed before any image checkpoint is saved. Runtime/API failures stop immediately with a traceback. Corrupt images are recorded separately and processing continues after smoke succeeds.
Progress includes images/second and ETA (resume hashing counts as images). Each result is flushed and atomically saved. Resume requires identical original bytes, profile and checkpoint integrity; corrupt-image errors are retried. All-corrupt inputs stop because there is no valid real image for smoke. Re-running this cell creates a fresh invocation; no old download links are accepted.

In [ ]:
RUN_OK = False
RUN_ID = uuid.uuid4().hex
EXPORT_OK = False
try:
    helpers['validate_publish'](root, OUTPUT, ENV_ROOT, ARCHIVE)
    helpers['invalidate_ready'](OUTPUT)
    subprocess.run([str(PYTHON), '-I', '-u', str(WORKER), 'run', str(root), str(OUTPUT),
                    str(ENV_ROOT), RUN_ID, '--archive', str(ARCHIVE)],
                   env=child_env, cwd=SCRATCH, check=True)
    RUN_OK = True
except BaseException:
    helpers['invalidate_current_ready'](root, OUTPUT, ENV_ROOT, RUN_ID)
    RUN_OK = False
    print('WARNING: run failed; an older ocr.zip may remain, not a current result.', flush=True)
    raise


## 4 · Publish only ocr.zip
The only final output is **`/kaggle/working/ocr.zip`**. Download and extract it to browse the records. Original filename padding is preserved: `L04_V001/001.jpg` → `L04/L04_V001/001.json` (not `1.json` or a hash). The group `L04` comes from canonical `video_id`, never ancestor/root guessing, even for a single-video input. Each JSON is a one-element array with exactly `video_id`, `frame_id` (null), `keyframe_n`, `image_path`, `ocr_text`, `retrieval_text`, and `status`. `image_path` is a caption-mapping-style logical reference such as `data/L04/L04_V001/001.jpg`, preserving the filename's padding and actual extension/case; no caption mapping is read or modified and no captions are generated. Nonblank model lines are joined with newlines in model order, without trimming retained text or extra confidence filtering/correction; retrieval text is identical. Successful status is `ok` for retained text or `empty` otherwise; decode errors stay `error` with empty text and no exception details. Technical checkpoints remain private. Inside the ZIP (only OCR JSON records; no summary, profile, wrapper, environment, cache or checkpoints):
```text
L04/
  L04_V001/
    001.json
  L04_V002/
    002.json
L05/
  L05_V001/
    001.json
```
Only current, integrity-verified records are exported; removed/stale checkpoints are excluded. Images (or folders) whose names end in a copy suffix like `001 (1).jpg` are silently skipped; otherwise canonical video folders and numeric image stems are required; duplicate identities (including `001`/`1` or JPEG/PNG with the same stem) and alias video folders are rejected before export writes. Check the summary printed in the cell 4 log (it is not in the ZIP): `complete` is false for any decode error or missing/invalid checkpoint. If the inventory changed, rerun cell 3. A unique temporary publication file may exist in working during packing/copying; it is removed on normal success or handled failure/interruption. ZIP validation, atomic replacement and a scratch-only invocation/profile/digest receipt gate the single link. Prepublication failures preserve the previous owned ZIP but never link it as current; unrelated preexisting ZIPs are refused. No unrelated working files are enumerated or cleaned: the only-ZIP guarantee assumes a fresh session. Resume is same-session only; transient state is not persisted across Kaggle resets. Worker-source changes invalidate old profile/checkpoints without weakening integrity.

In [ ]:
EXPORT_OK = False
try:
    if not RUN_OK:
        raise RuntimeError('Cell 3 did not finish successfully; no current downloads')
    helpers['validate_publish'](root, OUTPUT, ENV_ROOT, ARCHIVE)
    helpers['invalidate_ready'](OUTPUT)
    subprocess.run([str(PYTHON), '-I', '-u', str(WORKER), 'export', str(root), str(OUTPUT),
                    str(ENV_ROOT), RUN_ID, '--archive', str(ARCHIVE)],
                   env=child_env, cwd=SCRATCH, check=True)
    download = helpers['checked_ready'](root, OUTPUT, ENV_ROOT, RUN_ID, ARCHIVE)
    from IPython.display import FileLink, display
    display(FileLink(str(download)))  # The only downloadable output.
    EXPORT_OK = True
except BaseException:
    EXPORT_OK = False
    if RUN_OK:
        helpers['invalidate_current_ready'](root, OUTPUT, ENV_ROOT, RUN_ID)
    print('WARNING: export failed; an older ocr.zip may remain, not a current result.', flush=True)
    raise
